En este notebook se aplica la limpieza a la base de datos, basado en la exploracion anterior y se verifican resultados

Adicional, basado en la exploracion de los datos, la base de datos esta compuesta por:

1. customer_id : Corresponde al identificador por cliente en la organzacion.
2. full_name : Nombre completo del cliente
3. email : Correo electronico del cliente
4. phone : Numero telefonico del cliente
5. signup_date :' Fecha en el que el usuario inicio la suscripcion
6. last_purchase_date : Fecha de la ultima compra del cliente
7. monthly_spend : Gasto en el mes
8. total_shipments : Total de envios
9. churn_label : Tasa de abandono del cliente (Target)
10. home_address : Direccion del cliente

### Cargar librerias

In [1]:
import sys
from pathlib import Path  # Manejo de rutas de archivo 
#Encontrar la raiz del proyecto
ROOT = Path.cwd()
while not (ROOT / 'src').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT)) # Encuentra la carpeta src
#Se cargan las funciones creadas para limpieza
from src.function_clean import *

### Cargar dataframe 

In [2]:
#Ruta de la base de datos
ruta = "../database/bronze/raw_data/raw_data_customers.csv"

In [3]:
df = load_data(ruta) # Cargando los datos
df = stand_nulls(df) # estandarizando los valores nulos
df.head(5) # revision de los 5 valores

,customer_id,full_name,email,phone,signup_date,last_purchase_date,monthly_spend,total_shipments,churn_label,home_address
0,C001,Juan Perez,jperez@email.com,555-0101,2023-01-15,12/05/2025,450.5,12.0,0.0,Calle Falsa 123
1,C002,Maria Garcia,m.garcia@provider.net,555-0102,2023-02-20,2025-05-10,1200.0,45.0,0.0,Carrera 7 # 45-10
2,C003,Carlos Rodriguez,c.rod@work.com,NaN,2023-03-05,03/25/2025,NaN,8.0,1.0,Av. Siempre Viva 742
3,C004,Ana Martinez,ana.mtz@mail.com,555-0104,2023-04-12,2025-06-01,890.2,22.0,0.0,Clle 100 # 15
4,C001,Juan Perez,jperez@email.com,555-0101,2023-01-15,12/05/2025,450.5,12.0,0.0,Calle Falsa 123


### Tratamiento de fechas

In [4]:
# Limpieza de fechas
df["signup_date"] = type_dates(df, "signup_date")
df["last_purchase_date"] = type_dates(df, "last_purchase_date")
df.head()

,customer_id,full_name,email,phone,signup_date,last_purchase_date,monthly_spend,total_shipments,churn_label,home_address
0,C001,Juan Perez,jperez@email.com,555-0101,2023-01-15,2025-12-05,450.5,12.0,0.0,Calle Falsa 123
1,C002,Maria Garcia,m.garcia@provider.net,555-0102,2023-02-20,2025-05-10,1200.0,45.0,0.0,Carrera 7 # 45-10
2,C003,Carlos Rodriguez,c.rod@work.com,NaN,2023-03-05,2025-03-25,NaN,8.0,1.0,Av. Siempre Viva 742
3,C004,Ana Martinez,ana.mtz@mail.com,555-0104,2023-04-12,2025-06-01,890.2,22.0,0.0,Clle 100 # 15
4,C001,Juan Perez,jperez@email.com,555-0101,2023-01-15,2025-12-05,450.5,12.0,0.0,Calle Falsa 123


Se identificaron fechas almacenadas en diferentes formatos. Los registros con guiones siguen la estructura AAAA-MM-DD, mientras que en los registros con barras se encontraron los formatos MM/DD/AAAA y DD/MM/AAAA. Para diferenciarlos, cuando el primer valor es mayor que 12 se interpreta como el dia; en los casos ambiguos se asume el formato MM/DD/AAAA. Esta decision se apoyo en la presencia de fechas como 03/25/2025, que solo puede interpretarse como mes/dia/año. De esta manera, cada formato se procesa por separado para evitar el intercambio incorrecto entre el dia y el mes.

In [5]:
df[["signup_date", "last_purchase_date"]].isnull().sum()

signup_date           0
last_purchase_date    1
dtype: int64

In [6]:
#Verificando si existen fechas last_purchase_date menores a signup_date
inconsistentes = df[df["last_purchase_date"] < df["signup_date"]]
print(f"Filas con last_purchase_date < signup_date: {len(inconsistentes)}")
inconsistentes

Filas con last_purchase_date < signup_date: 2


,customer_id,full_name,email,phone,signup_date,last_purchase_date,monthly_spend,total_shipments,churn_label,home_address
47,C044,Iván Lalinde,i.lalinde@tv.co,555-0144,2025-03-01,2025-01-06,700.0,24.0,0.0,Calle 100 # 19
84,C081,Taliana Vargas,t.vargas@miss.co,555-0181,2025-05-01,2025-01-06,2200.0,75.0,0.0,Santa Marta


Se identificaron 2 registros en los que la fecha de la ultima compra (last_purchase_date) es anterior a la fecha de registro del cliente (signup_date), lo que representa una inconsistencia temporal, ya que un cliente no puede realizar una compra antes de haberse registrado. Al analizar estos casos, se observa que presentan la fecha 2025-06-10, lo que sugiere un posible error sistematico en el proceso de captura o carga de la informacion. Ademas, todos los registros pertenecen a clientes con churn_label = 0. Dado que no es posible determinar la fecha correcta de la ultima compra, se opta por reemplazar estos valores por nulos (NaN) para evitar introducir informacion incorrecta y prevenir la distorsion de variables derivadas que dependan de esta fecha, como la antigüedad del cliente

### Tratamiento de valores duplicados

In [7]:
print(f"registros antes de quitar duplicados: {df.shape[0]}")

registros antes de quitar duplicados: 114


In [8]:
# eliminacion de valores duplicados por customer_id
df = delete_duplicates(df, "customer_id")
df.head(5)

,customer_id,full_name,email,phone,signup_date,last_purchase_date,monthly_spend,total_shipments,churn_label,home_address
0,C001,Juan Perez,jperez@email.com,555-0101,2023-01-15,2025-12-05,450.5,12.0,0.0,Calle Falsa 123
1,C002,Maria Garcia,m.garcia@provider.net,555-0102,2023-02-20,2025-05-10,1200.0,45.0,0.0,Carrera 7 # 45-10
3,C004,Ana Martinez,ana.mtz@mail.com,555-0104,2023-04-12,2025-06-01,890.2,22.0,0.0,Clle 100 # 15
5,C005,Luis Herrera,lherrera@domain.org,555-0105,2023-05-30,2024-12-20,310.0,5.0,1.0,Apt 502 Torre B
6,C006,Sonia Castro,scastro@test.com,555-0106,2023-06-21,2025-04-15,2500.0,NaN,0.0,Calle 80 # 20


In [9]:
print(f"registros despues de quitar duplicados: {df.shape[0]}")

registros despues de quitar duplicados: 110


### Tratamiento de PII

In [10]:
#Se eliminar las variables sensibles y se anonimiza la variable customer_id
df , df_ids = anonymous_columns(df,"customer_id")

In [11]:
# se verifica el df sin PII
df.head()

,signup_date,last_purchase_date,monthly_spend,total_shipments,churn_label,customer_id_anonymous
0,2023-01-15,2025-12-05,450.5,12.0,0.0,C00000
1,2023-02-20,2025-05-10,1200.0,45.0,0.0,C00001
2,2023-04-12,2025-06-01,890.2,22.0,0.0,C00002
3,2023-05-30,2024-12-20,310.0,5.0,1.0,C00003
4,2023-06-21,2025-04-15,2500.0,NaN,0.0,C00004


In [12]:
#Se verifica el df que contiene los ids
df_ids.head()

,customer_id,customer_id_anonymous
0,C001,C00000
1,C002,C00001
2,C004,C00002
3,C005,C00003
4,C006,C00004


### Tratamiento de posibles inconsistencias en los datos

In [13]:
#Como se indentifico en la exploracion de los datos, existen registros que contienen inconsistencias tanto en monthly_spend como en total_shipments
# verificamos los valores menores o iguales a 0 de monthly_spend
df[df["monthly_spend"]<=0]

,signup_date,last_purchase_date,monthly_spend,total_shipments,churn_label,customer_id_anonymous
7,2023-08-14,2025-05-28,-50.0,15.0,0.0,C00007
30,2024-09-01,2025-01-01,-100.0,10.0,1.0,C00030
89,2023-09-12,2024-12-12,-10.0,2.0,1.0,C00089


In [14]:
# verificamos los valores mayores o iguales a 99999 de monthly_spend
df[df["monthly_spend"]>=99999]

,signup_date,last_purchase_date,monthly_spend,total_shipments,churn_label,customer_id_anonymous
6,2023-03-05,2025-03-25,99999.0,8.0,1.0,C00006
22,2024-05-20,2025-05-20,99999.0,50.0,0.0,C00022
62,2023-12-05,2025-05-06,99999.0,90.0,0.0,C00062


In [15]:
#Se procede a reemplazar los valores inconsistentes de monthly_spend por nulos
df["monthly_spend"]= mistake_(df, "monthly_spend")

In [16]:
#verificamos los valores menores o iguales a 0 de total_shipments
df[df["total_shipments"]<=0]

,signup_date,last_purchase_date,monthly_spend,total_shipments,churn_label,customer_id_anonymous
51,2023-01-01,2025-01-01,NaN,0.0,1.0,C00051
57,2023-07-01,2025-01-01,NaN,0.0,1.0,C00057


Se identificaron registros con total_shipments igual a cero y monthly_spend nulo. Aunque podria asumirse que un cliente sin envios presenta un gasto igual a cero, la presencia de una fecha de ultima compra (last_purchase_date) indica que el cliente ha realizado transacciones previamente. Dado que no se dispone de un diccionario de datos que confirme el periodo al que corresponde cada variable, no es posible asegurar que un valor de 0 sea correcto. Por esta razon, estos registros se mantendran como valores faltantes y seran tratados mediante imputacion junto con el resto de los datos nulos.

In [17]:
df[df["total_shipments"]>=1000]

,signup_date,last_purchase_date,monthly_spend,total_shipments,churn_label,customer_id_anonymous
9,2023-10-10,2024-10-10,420.0,1000.0,NaN,C00009


In [18]:
#Se procede a reemplazar los valores inconsistentes de shipments por nulos
df["total_shipments"]= mistake_(df, "total_shipments")

In [19]:
#Se procede a reemplazar los valores inconsistentes de la variable last_purchase_date
df["last_purchase_date"]= mistake_(df, "last_purchase_date")

### Valores nulos

In [20]:
# verificar valores nulos despues de haber quitado valores inconsistentes
df.isnull().mean().round(4)*100

signup_date               0.00
last_purchase_date        2.73
monthly_spend            19.09
total_shipments           2.73
churn_label               0.91
customer_id_anonymous     0.00
dtype: float64

In [21]:
# Verificar registros con churn_label = a null
df[df["churn_label"].isnull()]

,signup_date,last_purchase_date,monthly_spend,total_shipments,churn_label,customer_id_anonymous
9,2023-10-10,2024-10-10,420.0,NaN,NaN,C00009


In [22]:
#Eliminar valores nulos por variable objetivo
df = df.dropna(subset=["churn_label"])

In [23]:
df.shape #Cantidad de registros y columnas despues de eliminar duplicados y nulos por churn_label

(109, 6)

In [24]:
df.isnull().mean().round(4)*100 #Porcentaje de nulos por cada variable

signup_date               0.00
last_purchase_date        2.75
monthly_spend            19.27
total_shipments           1.83
churn_label               0.00
customer_id_anonymous     0.00
dtype: float64

Antes de eliminar o imputar los valores nulos, se realiza el analisis exploratorio de datos (EDA). Esto permite evaluar la cantidad de datos faltantes, identificar posibles patrones y determinar el tratamiento mas adecuado para cada variable. Eliminarlos directamente durante la limpieza podria generar perdida de informacion relevante y reducir innecesariamente la cantidad de registros disponibles

### Datos atipicos

In [25]:
#validar los registros atipicos cola derecha de shipments mediante el rango intercuartilico
Q1 = df["total_shipments"].quantile(0.25)
Q3 = df["total_shipments"].quantile(0.75)
IQR = Q3 - Q1
limite_inferior = Q1 - 1.5 * IQR
limite_superior = Q3 + 1.5 * IQR
df[df["total_shipments"] > limite_superior]
df.loc[df["total_shipments"] > limite_superior]


,signup_date,last_purchase_date,monthly_spend,total_shipments,churn_label,customer_id_anonymous
45,2025-03-20,2025-06-20,5000.0,150.0,0.0,C00045
47,2025-04-10,2025-10-06,3500.0,120.0,0.0,C00047
53,2023-03-15,2025-06-20,10000.0,400.0,0.0,C00053
61,2023-11-20,2025-06-20,7000.0,200.0,0.0,C00061
64,2024-05-26,2025-05-26,5000.0,200.0,0.0,C00064
66,2024-03-01,2025-01-06,6000.0,180.0,0.0,C00066
67,2024-04-12,2025-06-15,3200.0,115.0,0.0,C00067
70,2024-07-15,2025-06-15,8000.0,250.0,0.0,C00070
73,2024-10-25,2025-06-20,9500.0,350.0,0.0,C00073
78,2025-03-20,2025-06-20,4500.0,160.0,0.0,C00078


In [26]:
df[df["total_shipments"] < limite_inferior]
df.loc[df["total_shipments"] < limite_inferior]


,signup_date,last_purchase_date,monthly_spend,total_shipments,churn_label,customer_id_anonymous


In [27]:
#validar los registros atipicos cola derecha de monthly_spend
Q1 = df["monthly_spend"].quantile(0.25)
Q3 = df["monthly_spend"].quantile(0.75)
IQR = Q3 - Q1
limite_inferior = Q1 - 1.5 * IQR
limite_superior = Q3 + 1.5 * IQR
df[df["monthly_spend"] > limite_superior]
df.loc[df["monthly_spend"] > limite_superior]


,signup_date,last_purchase_date,monthly_spend,total_shipments,churn_label,customer_id_anonymous
45,2025-03-20,2025-06-20,5000.0,150.0,0.0,C00045
53,2023-03-15,2025-06-20,10000.0,400.0,0.0,C00053
61,2023-11-20,2025-06-20,7000.0,200.0,0.0,C00061
64,2024-05-26,2025-05-26,5000.0,200.0,0.0,C00064
66,2024-03-01,2025-01-06,6000.0,180.0,0.0,C00066
70,2024-07-15,2025-06-15,8000.0,250.0,0.0,C00070
73,2024-10-25,2025-06-20,9500.0,350.0,0.0,C00073
78,2025-03-20,2025-06-20,4500.0,160.0,0.0,C00078
85,2023-05-30,2025-05-30,4200.0,140.0,0.0,C00085
93,2024-01-20,2025-06-15,10000.0,500.0,0.0,C00093


In [28]:
df[df["monthly_spend"] < limite_inferior]
df.loc[df["monthly_spend"] < limite_inferior]

,signup_date,last_purchase_date,monthly_spend,total_shipments,churn_label,customer_id_anonymous


Se observa que los clientes con mayor cantidad de envios tambien presentan un mayor gasto mensual, por lo que existe coherencia entre ambas variables. Estos registros podrian representar clientes con una actividad comercial elevada y no necesariamente errores en los datos. Por esta razon, se conservaran inicialmente y se evaluara su efecto durante el modelado.

No se identificaron valores atipicos inferiores en monthly_spend ni en total_shipments. Esto se debe a que el limite inferior calculado mediante el rango intercuartílico se encuentra por debajo de los valores minimos observados. Por lo tanto, los valores atipicos de ambas variables se concentran unicamente en la cola derecha, correspondientes a clientes con mayor gasto y cantidad de envios.

In [29]:
#Ruta de la base de datos
ruta = "../database/silver/clean_data/clean_data_customers.csv"

In [30]:
guardar_dataframe(df,ruta)

Archivo guardado correctamente en: ..\database\silver\clean_data\clean_data_customers.csv


In [31]:
ruta_ids = "../database/secure/id_mapping/customer_ids.csv"

In [32]:
guardar_dataframe(df_ids,ruta_ids)

Archivo guardado correctamente en: ..\database\secure\id_mapping\customer_ids.csv
